In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

#### core vector funtions

In [8]:
def as_vector3(value):
    vector = np.asarray(value, dtype=float)
    return vector

def dot_product(a, b):
    a = as_vector3(a)
    b = as_vector3(b)
    return float(np.sum(a * b))

def vector_norm(vector):
    vector = as_vector3(vector)
    return float(np.sqrt(np.sum(vector * vector)))

def unit_vector(vector):
    vector = as_vector3(vector)
    norm = vector_norm(vector)
    if np.isclose(norm, 0.0):
        raise ValueError("A zero vector has no direction.")
    return vector / norm

def cross_product(a, b):
    a = as_vector3(a)
    b = as_vector3(b)
    return np.cross(a, b)

In [3]:
def distance_between_points(point_a, point_b):
    point_a = as_vector3(point_a)
    point_b = as_vector3(point_b)
    return vector_norm(point_b - point_a)

def angle_between(a, b):
    a = as_vector3(a)
    b = as_vector3(b)
    denominator =  vector_norm(a) * vector_norm(b)
    if np.isclose(denominator, 0.0):
        raise ValueError("The angle is undefined for a zero vector.")
    
    cosine = dot_product(a, b) / denominator
    cosine = np.clip(cosine, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

#### Define the drone mission

In [4]:
waypoint_names = ["Start", "A", "B", "C", "Finish"]
waypoints_world = np.array([
    [1.0, 1.0, 1.0],
    [4.0, 2.0, 3.0],
    [7.0, 6.0, 5.0],
    [5.0, 9.0, 7.0],
    [9.0, 11.0, 4.0]
])

print("World-coordinate waypoints:\n")
for name, point in zip(waypoint_names, waypoints_world):
    print(f"{name:>6}: {point}")

World-coordinate waypoints:

 Start: [1. 1. 1.]
     A: [4. 2. 3.]
     B: [7. 6. 5.]
     C: [5. 9. 7.]
Finish: [ 9. 11.  4.]


#### Calculate flight vectors and segment distances

In [10]:
flight_vectors = waypoints_world[1:] - waypoints_world[:-1]
segment_distances = np.array([vector_norm(vector) for vector in flight_vectors])

unit_directions = np.array([unit_vector(vector) for vector in flight_vectors])
total_distance = float(np.sum(segment_distances))

In [11]:
print("Flight segment analysis:\n")

for index, (vector, distance, direction) in enumerate(
    zip(flight_vectors, segment_distances, unit_directions)
):
    start_name = waypoint_names[index]
    end_name = waypoint_names[index + 1]
    print(f"{start_name} -> {end_name}")
    print(f"displacement : {vector}")
    print(f"distance     : {distance:.3f} m")
    print(f"unit vector  : {direction}")
    print()

print(f"Total mission distance: {total_distance:.3f} m")

Flight segment analysis:

Start -> A
displacement : [3. 1. 2.]
distance     : 3.742 m
unit vector  : [0.802 0.267 0.535]

A -> B
displacement : [3. 4. 2.]
distance     : 5.385 m
unit vector  : [0.557 0.743 0.371]

B -> C
displacement : [-2.  3.  2.]
distance     : 4.123 m
unit vector  : [-0.485  0.728  0.485]

C -> Finish
displacement : [ 4.  2. -3.]
distance     : 5.385 m
unit vector  : [ 0.743  0.371 -0.557]

Total mission distance: 18.635 m


#### Analyze the drone's turns

In [12]:
turn_angles = []
turn_normals = []

for index in range(len(flight_vectors) - 1):
    incoming = flight_vectors[index]
    outgoing = flight_vectors[index + 1]
    
    angle = angle_between(incoming, outgoing)
    normal = cross_product(incoming, outgoing)

    turn_angles.append(angle)
    turn_normals.append(normal)

    waypoint = waypoint_names[index + 1]
    print(f"Turn at waypoint {waypoint}")
    print(f"angle   : {angle:.3f} degrees")
    print(f"turning-plane normal: {normal}")
    
turn_angles = np.array(turn_angles)
turn_normals = np.array(turn_normals)

Turn at waypoint A
angle   : 32.468 degrees
turning-plane normal: [-6.  0.  9.]
Turn at waypoint B
angle   : 63.232 degrees
turning-plane normal: [  2. -10.  17.]
Turn at waypoint C
angle   : 111.119 degrees
turning-plane normal: [-13.   2. -16.]
